# Week 90, Day 2: Build

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w090d2.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


These are the files this page's programs use.

In [ ]:
# Run this cell first: it creates the files used below.
with open('waymark_domain.py', 'w', encoding='utf-8') as f:
    f.write('import heapq\n\n\nclass UnknownPlace(LookupError):\n    pass\n\n\nclass NoRoute(Exception):\n    pass\n\n\ndef naismith_minutes(km, climb_m):\n    return round(km / 5 * 60 + climb_m / 600 * 60)\n\n\nclass Route:\n    def __init__(self, stops, km, climb_m):\n        self.stops, self.km, self.climb_m = list(stops), km, climb_m\n\n    @property\n    def minutes(self):\n        return naismith_minutes(self.km, self.climb_m)\n\n    def __repr__(self):\n        return f"Route({self.stops}, {self.km} km, {self.climb_m} m)"\n\n\nclass PathNetwork:\n    def __init__(self, heights):\n        self.heights = dict(heights)\n        self.paths = {name: {} for name in self.heights}\n\n    def _check(self, *names):\n        for name in names:\n            if name not in self.heights:\n                raise UnknownPlace(name)\n\n    def add_path(self, a, b, km):\n        self._check(a, b)\n        if km <= 0:\n            raise ValueError("km must be positive")\n        self.paths[a][b] = km\n        self.paths[b][a] = km\n\n    def climb(self, stops):\n        return sum(max(0, self.heights[b] - self.heights[a]) for a, b in zip(stops, stops[1:]))\n\n    def shortest(self, start, end):\n        self._check(start, end)\n        dist, prev, done = {start: 0}, {}, set()\n        heap = [(0, start)]\n        while heap:\n            d, node = heapq.heappop(heap)\n            if node in done:\n                continue\n            done.add(node)\n            if node == end:\n                break\n            for nxt, km in sorted(self.paths[node].items()):\n                if d + km < dist.get(nxt, float("inf")):\n                    dist[nxt] = d + km\n                    prev[nxt] = node\n                    heapq.heappush(heap, (d + km, nxt))\n        if end not in done:\n            raise NoRoute(f"no route from {start} to {end}")\n        stops = [end]\n        while stops[-1] != start:\n            stops.append(prev[stops[-1]])\n        stops.reverse()\n        return Route(stops, round(dist[end], 2), self.climb(stops))\n')
with open('waymark_map.py', 'w', encoding='utf-8') as f:
    f.write('from waymark_domain import PathNetwork\n\nHEIGHTS = {"Mill": 40, "Weir": 55, "Church": 80, "Ridge": 310, "Farm": 120, "Wood": 150, "Lake": 60, "Island": 5}\nPATHS = [("Mill", "Weir", 1.5), ("Weir", "Church", 2.0), ("Mill", "Church", 4.0), ("Church", "Ridge", 3.0),\n         ("Weir", "Farm", 2.5), ("Farm", "Ridge", 2.0), ("Farm", "Wood", 1.5), ("Wood", "Lake", 2.0), ("Lake", "Mill", 3.5)]\n\n\ndef sample_network():\n    network = PathNetwork(HEIGHTS)\n    for a, b, km in PATHS:\n        network.add_path(a, b, km)\n    return network\n')
with open('waymark_geo.py', 'w', encoding='utf-8') as f:
    f.write('import math\nfrom waymark_domain import PathNetwork\nfrom waymark_map import HEIGHTS, PATHS\n\nPOSITIONS = {"Mill": (0.0, 0.0), "Weir": (1.4, 0.4), "Church": (2.8, 1.2), "Ridge": (4.4, 3.4), "Farm": (2.8, 2.4),\n             "Wood": (2.0, 3.5), "Lake": (0.6, 2.8), "Island": (-3.0, 5.0)}\n\n\nclass GeoNetwork(PathNetwork):\n    """A path network whose junctions also have positions, in kilometres east and north."""\n\n    def __init__(self, heights, positions):\n        super().__init__(heights)\n        self.positions = dict(positions)\n\n    def straight_km(self, a, b):\n        return math.dist(self.positions[a], self.positions[b])\n\n\ndef sample_geo():\n    network = GeoNetwork(HEIGHTS, POSITIONS)\n    for a, b, km in PATHS:\n        network.add_path(a, b, km)\n    return network\n\n\ndef grid_network(size):\n    """A size-by-size grid of junctions named \'r-c\', 1 km apart, with slightly longer paths to their neighbours."""\n    names = {(r, c): f"{r}-{c}" for r in range(size) for c in range(size)}\n    heights = {name: ((r * 13 + c * 7) % 9) * 20 for (r, c), name in names.items()}\n    positions = {name: (float(c), float(r)) for (r, c), name in names.items()}\n    network = GeoNetwork(heights, positions)\n    for (r, c), name in names.items():\n        if c + 1 < size:\n            network.add_path(name, names[(r, c + 1)], 1.0 + ((r * 7 + c * 3) % 5) / 10)\n        if r + 1 < size:\n            network.add_path(name, names[(r + 1, c)], 1.0 + ((r * 3 + c * 5) % 5) / 10)\n    return network\n')

**Predict before you run:** what will this print?



In [ ]:
import base64
import hashlib
import math

raw = bytes(range(10))
code = base64.b32encode(raw).decode().rstrip("=").lower()
print(code, len(code))
print(round(len(code) * math.log2(32)), "bits")
print(hashlib.sha256(code.encode()).hexdigest()[:16])

> **Key idea.** Share codes are bearer credentials: make them from `secrets` (inject the byte source for tests), with enough entropy (`length × log2(alphabet)` bits; 80 is plenty), in a readable alphabet such as lower-case base32. Store only SHA-256 hashes of codes, with expiry, revocation and view counts. Give unknown, expired and revoked codes the same 404. Export routes as GeoJSON Features with `LineString` geometry in `[x, y]` order and properties; validate incoming GeoJSON's structure, numbers and that every point is on the map.

### Exercise: Share codes

Write `make_code(random_bytes, nbytes=10)`: call `random_bytes(nbytes)` and return its base32 encoding in lower case without `=` padding. Then `entropy_bits(length, alphabet_size)`: the bits of entropy in a random code of that length, rounded to 1 place, and `guesses_needed(bits, codes_in_use)`: the expected number of random guesses to hit any valid code, `2 ** bits / codes_in_use`, as a whole number.

In [ ]:
import base64
import math

def make_code(random_bytes, nbytes=10):
    ___

def entropy_bits(length, alphabet_size):
    ___

def guesses_needed(bits, codes_in_use):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import secrets\n\ndef test_codes():\n    \"readable and unguessable\"\n    assert make_code(lambda n: bytes(range(n))) == \"aaaqeayeaudaocajbifq\"[:16]\n    code = make_code(secrets.token_bytes)\n    assert len(code) == 16 and code == code.lower() and \"=\" not in code\n    assert make_code(lambda n: b\"\\xff\" * n, 5) == \"77777777\"\n    assert entropy_bits(16, 32) == 80.0 and entropy_bits(6, 10) == 19.9\n    assert guesses_needed(20, 1000) == 1048 and guesses_needed(80, 10 ** 6) > 10 ** 18")

### Exercise: A share store

Write `ShareStore(clock)`, where `clock()` returns the time in seconds. `create(route, days, code)` stores a share of `route` that expires `days × 86400` seconds from now, keyed by the SHA-256 hex digest of `code` (never the code itself), and returns the expiry time. `open(code)` returns the route and adds one to the share's views, or `None` if the code is unknown, revoked or expired (`now >= expiry`). `revoke(code)` returns whether there was an unrevoked share. `views(code)` returns its view count (or `None`). The store must not hold any code as given.

In [ ]:
import hashlib

class ShareStore:
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("class Clock:\n    def __init__(self):\n        self.t = 1_000_000\n    def __call__(self):\n        return self.t\n\ndef test_store():\n    \"hashed, expiring, revocable\"\n    clock = Clock()\n    store = ShareStore(clock)\n    assert store.create(\"Ridge walk\", 7, \"mfrggzdfmztwq2lk\") == 1_000_000 + 7 * 86400\n    assert \"mfrggzdfmztwq2lk\" not in repr(vars(store))\n    assert store.open(\"mfrggzdfmztwq2lk\") == \"Ridge walk\" and store.open(\"mfrggzdfmztwq2lk\") == \"Ridge walk\"\n    assert store.views(\"mfrggzdfmztwq2lk\") == 2 and store.open(\"nottherightcode1\") is None and store.views(\"x\") is None\n    clock.t += 7 * 86400 - 1\n    assert store.open(\"mfrggzdfmztwq2lk\") == \"Ridge walk\"\n    clock.t += 1\n    assert store.open(\"mfrggzdfmztwq2lk\") is None, \"expired at exactly seven days\"\n    store.create(\"Lake loop\", 1, \"secondcode000000\")\n    assert store.revoke(\"secondcode000000\") is True and store.revoke(\"secondcode000000\") is False\n    assert store.open(\"secondcode000000\") is None")

### Exercise: GeoJSON out

Using the page's `waymark_geo`, write `to_geojson(network, name, stops)`: a dict for a GeoJSON `Feature` whose `geometry` is a `LineString` with `coordinates` the `[x, y]` positions of the stops (as lists), and whose `properties` are `name`, `stops`, `km` (1 place), `climb_m` and `minutes`. Then `to_text(feature)`: the JSON text with `sort_keys=True`.

In [ ]:
import json
from waymark_domain import Route
from waymark_geo import sample_geo

def to_geojson(network, name, stops):
    ___

def to_text(feature):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import json\n\ndef test_out():\n    \"a Feature with a LineString\"\n    f = to_geojson(sample_geo(), \"Ridge walk\", [\"Mill\", \"Weir\", \"Farm\", \"Ridge\"])\n    assert f == {\"type\": \"Feature\", \"geometry\": {\"type\": \"LineString\", \"coordinates\": [[0.0, 0.0], [1.4, 0.4], [2.8, 2.4], [4.4, 3.4]]},\n                 \"properties\": {\"name\": \"Ridge walk\", \"stops\": [\"Mill\", \"Weir\", \"Farm\", \"Ridge\"], \"km\": 6.0, \"climb_m\": 270, \"minutes\": 99}}\n    text = to_text(f)\n    assert json.loads(text) == f and text.startswith('{\"geometry\": ') and text.endswith('\"type\": \"Feature\"}'), \"keys sorted\"")

### Exercise: GeoJSON in

Write `read_line(text)`: parse JSON text and return the list of `(x, y)` tuples of a `Feature` whose geometry is a `LineString`. Raise `ValueError` with: `"not JSON"`; `"expected a Feature"`; `"expected a LineString"`; `"need at least two points"`; `"point N is not two numbers"` (N from 0; booleans aren't numbers). Then `junctions_at(positions, points, tolerance=0.05)`: the name of the junction within `tolerance` km of each point (the nearest if several), or `ValueError("point N is not at a junction")`.

In [ ]:
import json
import math

def read_line(text):
    ___

def junctions_at(positions, points, tolerance=0.05):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import json\nfrom waymark_geo import POSITIONS\n\ndef _feature(coords, geometry=\"LineString\", kind=\"Feature\"):\n    return json.dumps({\"type\": kind, \"geometry\": {\"type\": geometry, \"coordinates\": coords}, \"properties\": {}})\n\ndef test_in():\n    \"untrusted input, checked\"\n    assert read_line(_feature([[0, 0], [1.4, 0.4]])) == [(0, 0), (1.4, 0.4)]\n    for text, message in ((\"{nope\", \"not JSON\"), (_feature([[0, 0]], kind=\"FeatureCollection\"), \"expected a Feature\"),\n                          (_feature([[0, 0], [1, 1]], geometry=\"Point\"), \"expected a LineString\"),\n                          (_feature([[0, 0]]), \"need at least two points\"), (_feature([[0, 0], [1, True]]), \"point 1 is not two numbers\"),\n                          (_feature([[0, 0], [1, 2, 3]]), \"point 1 is not two numbers\")):\n        try:\n            read_line(text)\n        except ValueError as e:\n            assert str(e) == message, str(e)\n        else:\n            raise AssertionError(message)\n    assert junctions_at(POSITIONS, [(0.0, 0.01), (1.4, 0.4), (4.42, 3.39)]) == [\"Mill\", \"Weir\", \"Ridge\"]\n    try:\n        junctions_at(POSITIONS, [(0, 0), (2.0, 2.0)])\n    except ValueError as e:\n        assert str(e) == \"point 1 is not at a junction\"")

### Bug hunt: A route drawn in the sea

GeoJSON positions are `[x, y]` (longitude first), but this writes `[y, x]`, so the route appears mirrored across the diagonal: it prints `[[0.0, 0.0], [0.4, 1.4]]`. Write them in the right order, so it prints `[[0.0, 0.0], [1.4, 0.4]]`.

In [ ]:
positions = {"Mill": (0.0, 0.0), "Weir": (1.4, 0.4)}
coords = [[positions[s][1], positions[s][0]] for s in ["Mill", "Weir"]]
print(coords)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"x first\"\n    expect_output(\"[[0.0, 0.0], [1.4, 0.4]]\")")

### Bug hunt: A link that lasted a second too long

The share should stop working **at** its expiry time, but the check uses `>`, so at exactly that moment it still opens: it prints `open at expiry: True`. Treat the expiry moment as expired, so it prints `open at expiry: False`.

In [ ]:
expires = 1_604_800
def is_open(now):
    return not now > expires
print("open at expiry:", is_open(1_604_800))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"now >= expiry is expired\"\n    expect_output(\"open at expiry: False\")")

### Puzzle: Opening a shared link

Arrange the public handler: look the code up, and give the same 404 for every kind of failure. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
import math
p = ___   # bits of entropy in a 12-character code from a 32-letter alphabet
q = ___   # the first number in a GeoJSON position: "x (longitude)" or "y (latitude)"
r = ___   # True or False: a share code should be stored as given, so it can be shown again later
s = ___   # the status for an expired share link, from the public endpoint
t = ___   # the module to make share codes with: "random" or "secrets"

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p == 60 and q == \"x (longitude)\" and r is False, \"p, q and r\"\n    assert s == 404 and t == \"secrets\", \"s and t\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: the sharing API

Write `SharingAPI(clock, random_bytes, routes, network)` using the given helpers, where `routes` maps saved route names to stops. `handle(method, path, body)` returns `(status, data)`:

- `POST /shares` with JSON `{"route": NAME, "days": N}`: `400 {"error": "days must be 1 to 30"}` unless `days` is a whole number from 1 to 30 (not a boolean); `404 {"error": "no such route"}` for an unknown route; otherwise `201 {"code", "url", "expires"}` with a new code from `make_code(random_bytes)`, the URL `"/s/" + code` and the expiry time.
- `GET /s/CODE`: `200` with the route's GeoJSON Feature (from `to_geojson`) and `properties` also including `"views"` (after counting this one); otherwise `404 {"error": "this link doesn't work"}`.
- `DELETE /shares/CODE`: `204 None` if a live share was revoked, else `404 {"error": "no such share"}`.
- Bad JSON is `400 {"error": "bad request"}`; any other path `404 {"error": "not found"}`.

In [ ]:
import base64
import hashlib
import json
from waymark_domain import Route
from waymark_geo import sample_geo

def make_code(random_bytes, nbytes=10):
    return base64.b32encode(random_bytes(nbytes)).decode().rstrip("=").lower()

class ShareStore:
    def __init__(self, clock):
        self.clock, self.shares = clock, {}
    @staticmethod
    def _key(code):
        return hashlib.sha256(code.encode()).hexdigest()
    def create(self, route, days, code):
        expires = self.clock() + days * 86400
        self.shares[self._key(code)] = {"route": route, "expires": expires, "revoked": False, "views": 0}
        return expires
    def open(self, code):
        share = self.shares.get(self._key(code))
        if share is None or share["revoked"] or self.clock() >= share["expires"]:
            return None
        share["views"] += 1
        return share["route"]
    def revoke(self, code):
        share = self.shares.get(self._key(code))
        if share is None or share["revoked"] or self.clock() >= share["expires"]:
            return False
        share["revoked"] = True
        return True
    def views(self, code):
        share = self.shares.get(self._key(code))
        return None if share is None else share["views"]

def to_geojson(network, name, stops):
    km = sum(network.paths[a][b] for a, b in zip(stops, stops[1:]))
    route = Route(stops, km, network.climb(stops))
    return {"type": "Feature", "geometry": {"type": "LineString", "coordinates": [list(network.positions[s]) for s in stops]},
            "properties": {"name": name, "stops": list(stops), "km": round(km, 1), "climb_m": route.climb_m, "minutes": route.minutes}}

class SharingAPI:
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import json\nfrom waymark_geo import sample_geo\n\nclass Clock:\n    def __init__(self):\n        self.t = 1_000_000\n    def __call__(self):\n        return self.t\n\ndef _api():\n    counter = iter(range(1, 100))\n    random_bytes = lambda n: bytes([next(counter)]) * n\n    clock = Clock()\n    return SharingAPI(clock, random_bytes, {\"Ridge walk\": [\"Mill\", \"Weir\", \"Farm\", \"Ridge\"]}, sample_geo()), clock\n\ndef test_share_and_open():\n    \"create, open, count\"\n    api, clock = _api()\n    status, data = api.handle(\"POST\", \"/shares\", json.dumps({\"route\": \"Ridge walk\", \"days\": 7}))\n    assert status == 201 and data == {\"code\": \"aeaqcaibaeaqcaib\", \"url\": \"/s/aeaqcaibaeaqcaib\", \"expires\": 1_000_000 + 7 * 86400}\n    status, feature = api.handle(\"GET\", \"/s/aeaqcaibaeaqcaib\", \"\")\n    assert status == 200 and feature[\"geometry\"][\"coordinates\"][-1] == [4.4, 3.4] and feature[\"properties\"][\"views\"] == 1\n    assert api.handle(\"GET\", \"/s/aeaqcaibaeaqcaib\", \"\")[1][\"properties\"][\"views\"] == 2\n    clock.t += 7 * 86400\n    assert api.handle(\"GET\", \"/s/aeaqcaibaeaqcaib\", \"\") == (404, {\"error\": \"this link doesn't work\"})\n\ndef test_errors_and_revoke():\n    \"bad input, unknown routes, revocation\"\n    api, clock = _api()\n    for body in (json.dumps({\"route\": \"Ridge walk\", \"days\": 0}), json.dumps({\"route\": \"Ridge walk\", \"days\": 31}),\n                 json.dumps({\"route\": \"Ridge walk\", \"days\": True}), json.dumps({\"route\": \"Ridge walk\", \"days\": \"7\"})):\n        assert api.handle(\"POST\", \"/shares\", body) == (400, {\"error\": \"days must be 1 to 30\"}), body\n    assert api.handle(\"POST\", \"/shares\", json.dumps({\"route\": \"Moon walk\", \"days\": 3})) == (404, {\"error\": \"no such route\"})\n    assert api.handle(\"POST\", \"/shares\", \"{oops\") == (400, {\"error\": \"bad request\"})\n    code = api.handle(\"POST\", \"/shares\", json.dumps({\"route\": \"Ridge walk\", \"days\": 3}))[1][\"code\"]\n    assert api.handle(\"DELETE\", f\"/shares/{code}\", \"\") == (204, None)\n    assert api.handle(\"DELETE\", f\"/shares/{code}\", \"\") == (404, {\"error\": \"no such share\"})\n    assert api.handle(\"GET\", f\"/s/{code}\", \"\") == (404, {\"error\": \"this link doesn't work\"})\n    assert api.handle(\"GET\", \"/s/nonsense\", \"\")[0] == 404 and api.handle(\"PUT\", \"/shares\", \"\")[0] == 404")

## Warm-up

Ten random bytes (80 bits) make a 16-character code from a 32-letter alphabet. To guess one valid code out of a million in use, an attacker would need around `2**80 / 10**6` attempts, more than a billion billion: the code is the key, so it must come from `secrets`, never from `random`.

## Codes nobody can guess

A share link like `waymark.example/s/mfrggzdfmztwq2lk` is a **bearer** credential: whoever has it can open the route. So the code must be:

- **random from a secure source**: `secrets.token_bytes(10)` (Week 79), never `random`, whose sequence can be predicted from a few outputs, and never a counter or a hash of the route name;
- **long enough**: entropy is `length × log2(alphabet size)` bits; 80 bits is plenty for links that expire in days;
- **easy to type and read**: base32 in lower case avoids confusable characters and case problems in URLs.

For testing, inject the source of random bytes (as with clocks): production passes `secrets.token_bytes`, tests pass something predictable.

## Storing shares safely

Store each share as the **SHA-256 hash** of its code (like the reset tokens of Week 85), with the route, the time it was created, its expiry and whether it's been revoked. A leaked database then gives an attacker no working links. Looking a code up means hashing it and finding the hash.

Answer every failure the same way: unknown, expired and revoked codes all get the same `404` with the same message. Different answers would let an attacker learn which codes once existed. The owner, who is signed in, can list their shares and revoke them; viewers can't.

## GeoJSON out and in

**GeoJSON** (an open standard, RFC 7946) represents shapes as JSON. A route is a `Feature` whose `geometry` is a `LineString`: a list of `[x, y]` positions (in real GeoJSON, `[longitude, latitude]`, **in that order**, a common source of maps drawn in the sea), and whose `properties` hold anything else (name, distance, climb, time). Waymark's positions are kilometres on a local grid rather than longitude and latitude, so its export is GeoJSON in shape for the sake of the exercise; a real export would convert them.

Coming **in** (a walker importing a route someone sent), GeoJSON is untrusted input: check the `type`s, that `coordinates` is a list of at least two pairs of numbers, and then that each point is actually at a junction on the map (within a small tolerance), before treating it as a route.

## The public endpoints

| Method | Path | Who | Result |
|---|---|---|---|
| POST | /shares | owner | 201 with the code, URL and expiry; 400 for a bad number of days; 404 for an unknown route |
| GET | /s/CODE | anyone with the code | 200 with the GeoJSON; 404 otherwise |
| DELETE | /shares/CODE | owner | 204; 404 if there's no such share |

## Practice

## Quiz

## Daily challenge

## Your capstone today

1. **Anything public?** List every way someone without an account can reach your capstone. For each, what do they learn from an error?
2. **Secrets**: anything that acts as a key (links, tokens, codes) comes from `secrets`, is stored hashed and expires.
3. **An open format**: if your data has a standard format (GeoJSON, CSV, iCalendar, JSON Feed), export to it, and validate what comes back in.
4. **Merge** and update the burndown.

## Before you go

Tomorrow completes the API: consistent errors, pagination, versioning, and request checking in one place.